In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!pip -q install ultralytics opencv-python

In [7]:
# ============================================================
# 14_demo_video.ipynb
#
# Genera un vídeo de demo del pipeline completo:
#   Stage 1 — Detección de arma  (bbox amarillo)
#   Stage 2 — Pose estimation    (esqueleto + keypoints)
#   Stage 3 — Filtro temporal    (label AIMING/HOLDING/NEUTRAL)
#
# Clip objetivo: positivo con AIMING bien visible
# ============================================================

import os
import shutil
import math
from pathlib import Path
from collections import deque

import cv2
import numpy as np
from ultralytics import YOLO

print('OK Imports')

OK Imports


In [14]:
# ============================================================
# CONFIG
# ============================================================

# Clip a procesar — elige uno con AIMING bien visible
# Sugerencias basadas en los resultados del notebook 12:
#   PAH1_C1_P2_V1_HB_3  (133 frames arma, AIMING dominante)
#   PAH2_C1_P2_V1_HB_3  (172 frames arma)
#   PAH5_C1_P2_V1_HB_1  (182 frames arma)
CLIP_ID = 'PAH1_C1_P2_V1_HB_3'
GAR_ROOT = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset'
# Ruta al vídeo dentro del dataset GAR
# Estructura típica: GAR_ROOT / <action> / <clip_id> / video.mp4
# Ajusta VIDEO_PATH si la estructura es diferente
VIDEO_PATH = f'{GAR_ROOT}/Handgun/{CLIP_ID}/video.mp4'

OUT_DIR = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/demo'

WEAPON_DET_WEIGHTS = '/content/drive/MyDrive/TFM/experiments/weapon_det/yolov8m_weapons_B_e50_640/weights/best.pt'
POSE_WEIGHTS       = 'yolov8m-pose.pt'

# Inferencia
IMG_SIZE    = 640
CONF_WEAPON = 0.25
IOU_NMS     = 0.7
CONF_POSE   = 0.5
MAX_SECONDS = 15

# Filtro temporal
KP_CONF_THR      = 0.3
AIMING_ANGLE_THR = 160
POSE_WINDOW      = 10
AIMING_RUN_THR   = 3

# Vídeo de salida
OUTPUT_FPS    = 25
OUTPUT_WIDTH  = 1280   # se reescala manteniendo aspecto

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'OK Config — clip: {CLIP_ID}')

OK Config — clip: PAH1_C1_P2_V1_HB_3


In [9]:
# ============================================================
# KEYPOINTS COCO + HELPERS
# ============================================================

L_SHOULDER, R_SHOULDER = 5, 6
L_ELBOW,    R_ELBOW    = 7, 8
L_WRIST,    R_WRIST    = 9, 10

SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),
    (5,7),(7,9),
    (6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),
    (12,14),(14,16),
]

SKEL_COLOR_MAP = {
    (0,1):(255,200,0),(0,2):(255,200,0),(1,3):(255,200,0),(2,4):(255,200,0),
    (5,6):(0,255,100),
    (5,7):(0,200,255),(7,9):(0,200,255),
    (6,8):(0,200,255),(8,10):(0,200,255),
    (5,11):(0,255,100),(6,12):(0,255,100),(11,12):(0,255,100),
    (11,13):(200,0,255),(13,15):(200,0,255),
    (12,14):(200,0,255),(14,16):(200,0,255),
}

# Colores por label de pose (BGR)
POSE_COLORS = {
    'AIMING':  (0,   0,   255),   # rojo
    'HOLDING': (0,   165, 255),   # naranja
    'NEUTRAL': (0,   200, 0  ),   # verde
}

def angle_between(a, b, c):
    ba = (a[0]-b[0], a[1]-b[1])
    bc = (c[0]-b[0], c[1]-b[1])
    dot  = ba[0]*bc[0] + ba[1]*bc[1]
    norm = math.sqrt(ba[0]**2+ba[1]**2) * math.sqrt(bc[0]**2+bc[1]**2)
    if norm < 1e-6:
        return None
    return math.degrees(math.acos(max(-1.0, min(1.0, dot/norm))))

def get_arm_angles(kps, confs):
    results = {'left': None, 'right': None}
    for side, (sh, el, wr) in [('left',  (L_SHOULDER, L_ELBOW, L_WRIST)),
                                 ('right', (R_SHOULDER, R_ELBOW, R_WRIST))]:
        if all(confs[i] >= KP_CONF_THR for i in [sh, el, wr]):
            results[side] = angle_between(tuple(kps[sh]), tuple(kps[el]), tuple(kps[wr]))
    return results

def classify_pose(arm_angles):
    valid = {k: v for k, v in arm_angles.items() if v is not None}
    if not valid:
        return 'NEUTRAL'
    if any(v >= AIMING_ANGLE_THR for v in valid.values()):
        return 'AIMING'
    return 'HOLDING'

def bbox_center(box):
    return ((box[0]+box[2])/2, (box[1]+box[3])/2)

def find_nearest_person(weapon_box, person_boxes):
    if not person_boxes:
        return None
    wc = bbox_center(weapon_box)
    return int(np.argmin([math.dist(wc, bbox_center(pb)) for pb in person_boxes]))

print('OK Helpers')

OK Helpers


In [10]:
# ============================================================
# HELPERS DE ANOTACIÓN VISUAL
# ============================================================

def draw_skeleton(frame, kps, confs, highlight_arms=False):
    """Dibuja esqueleto y keypoints. Destaca brazos si highlight_arms=True."""
    for (i, j) in SKELETON:
        if confs[i] >= KP_CONF_THR and confs[j] >= KP_CONF_THR:
            pt1 = (int(kps[i][0]), int(kps[i][1]))
            pt2 = (int(kps[j][0]), int(kps[j][1]))
            color     = SKEL_COLOR_MAP.get((i,j), (200,200,200))
            thickness = 3 if highlight_arms and i in [5,6,7,8,9,10] else 2
            cv2.line(frame, pt1, pt2, color, thickness)
    for idx, (x, y) in enumerate(kps):
        if confs[idx] >= KP_CONF_THR:
            r = 6 if idx in [L_WRIST, R_WRIST] else 4
            cv2.circle(frame, (int(x), int(y)), r, (255,255,255), -1)
            cv2.circle(frame, (int(x), int(y)), r, (50,50,50), 1)


def draw_weapon_bbox(frame, x1, y1, x2, y2, conf):
    """BBox del arma en amarillo con etiqueta."""
    cv2.rectangle(frame, (int(x1),int(y1)), (int(x2),int(y2)), (0,255,255), 2)
    label = f'weapon {conf:.2f}'
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
    cv2.rectangle(frame, (int(x1), int(y1)-th-6), (int(x1)+tw+4, int(y1)), (0,255,255), -1)
    cv2.putText(frame, label, (int(x1)+2, int(y1)-4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,0), 1)


def draw_pose_label(frame, kps, confs, label):
    """Label de pose sobre la cabeza de la persona."""
    color = POSE_COLORS.get(label, (200,200,200))
    # Usar nariz o primer keypoint visible como referencia
    head_x, head_y = int(kps[0][0]), int(kps[0][1])
    if confs[0] < KP_CONF_THR:
        # Fallback: usar hombro
        for idx in [L_SHOULDER, R_SHOULDER]:
            if confs[idx] >= KP_CONF_THR:
                head_x, head_y = int(kps[idx][0]), int(kps[idx][1]) - 30
                break
    head_y = max(head_y - 18, 20)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(frame,
                  (head_x - 4, head_y - th - 4),
                  (head_x + tw + 4, head_y + 4),
                  color, -1)
    cv2.putText(frame, label, (head_x, head_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)


def draw_threat_overlay(frame, threat_confirmed, frame_w, frame_h):
    """Marco rojo parpadeante + texto THREAT CONFIRMED cuando se activa."""
    if not threat_confirmed:
        return
    cv2.rectangle(frame, (0,0), (frame_w-1, frame_h-1), (0,0,220), 6)
    label = 'THREAT CONFIRMED'
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.1, 3)
    cx = (frame_w - tw) // 2
    cv2.rectangle(frame, (cx-8, frame_h-th-20), (cx+tw+8, frame_h-4), (0,0,180), -1)
    cv2.putText(frame, label, (cx, frame_h-8),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255,255,255), 3)


def draw_hud(frame, frame_idx, total_frames, n_weapon_frames,
             pose_label, aiming_in_window, threat_confirmed, frame_w):
    """
    HUD minimalista en esquina superior izquierda:
    - Clip ID y progreso
    - Stage 1: weapon detection
    - Stage 2: pose label
    - Stage 3: aiming count en ventana
    """
    overlay = frame.copy()
    cv2.rectangle(overlay, (0,0), (300, 110), (20,20,20), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    def put(text, y, color=(220,220,220), scale=0.48, thick=1):
        cv2.putText(frame, text, (8, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thick)

    put(f'Frame {frame_idx:>4} / {total_frames}', 18)
    wcolor = (0,255,255) if n_weapon_frames > 0 else (120,120,120)
    put(f'S1 weapon : {"DETECTED" if n_weapon_frames > 0 else "none"}', 38, wcolor)
    pcolor = POSE_COLORS.get(pose_label, (180,180,180))
    put(f'S2 pose   : {pose_label}', 58, pcolor)
    acolor = (0,0,220) if threat_confirmed else (180,180,180)
    put(f'S3 aiming : {aiming_in_window}/{AIMING_RUN_THR} (win={POSE_WINDOW})', 78, acolor)
    tcolor = (0,0,220) if threat_confirmed else (80,180,80)
    put(f'THREAT    : {"YES" if threat_confirmed else "no"}', 100, tcolor, scale=0.52, thick=2)


print('OK Drawing helpers')

OK Drawing helpers


In [11]:
LOCAL_WEAPON_WEIGHTS = '/content/weapon_best.pt'
if not os.path.exists(LOCAL_WEAPON_WEIGHTS):
    print('Copiando pesos desde Drive...')
    shutil.copy2(WEAPON_DET_WEIGHTS, LOCAL_WEAPON_WEIGHTS)

weapon_model = YOLO(LOCAL_WEAPON_WEIGHTS)
pose_model   = YOLO(POSE_WEIGHTS)
print('OK Modelos cargados')

Copiando pesos desde Drive...
OK Modelos cargados


In [15]:
# ============================================================
# RENDER DEL VÍDEO DE DEMO
# ============================================================

local_in  = '/content/tmp_demo_in.mp4'
local_out = '/content/tmp_demo_out.mp4'
shutil.copy2(VIDEO_PATH, local_in)

cap      = cv2.VideoCapture(local_in)
fps_src  = cap.get(cv2.CAP_PROP_FPS) or 30
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w_src    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h_src    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
max_f    = min(n_frames, int(MAX_SECONDS * fps_src))

# Escalar manteniendo aspecto
scale   = OUTPUT_WIDTH / w_src
out_w   = OUTPUT_WIDTH
out_h   = int(h_src * scale)
if out_h % 2 != 0: out_h += 1   # ffmpeg requiere dimensiones pares

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(local_out, fourcc, OUTPUT_FPS, (out_w, out_h))

pose_window      = deque(maxlen=POSE_WINDOW)
threat_confirmed = False
n_weapon_frames  = 0
frame_i          = 0

print(f'Renderizando {max_f} frames ({max_f/fps_src:.1f}s) -> {out_w}x{out_h} @ {OUTPUT_FPS}fps')

while frame_i < max_f:
    ok, frame = cap.read()
    if not ok:
        break

    annotated   = frame.copy()
    pose_label  = 'NEUTRAL'
    aiming_in_w = 0
    has_weapon  = False

    # --- Stage 1: detección de arma ---
    wr = weapon_model.predict(frame, imgsz=IMG_SIZE, conf=CONF_WEAPON,
                               iou=IOU_NMS, verbose=False, device='cuda')[0]
    weapon_boxes = []
    if wr.boxes is not None and len(wr.boxes) > 0:
        for b in wr.boxes:
            x1,y1,x2,y2 = map(float, b.xyxy[0])
            conf = float(b.conf[0])
            weapon_boxes.append((x1,y1,x2,y2,conf))
            draw_weapon_bbox(annotated, x1, y1, x2, y2, conf)
        has_weapon = True
        n_weapon_frames += 1

    # --- Stage 2: pose estimation (solo si hay arma) ---
    if weapon_boxes:
        pr = pose_model.predict(frame, imgsz=IMG_SIZE, conf=CONF_POSE,
                                 verbose=False, device='cuda')[0]
        person_boxes_raw = []

        if pr.keypoints is not None and len(pr.keypoints) > 0:
            kps_data  = pr.keypoints.xy.cpu().numpy()
            conf_data = pr.keypoints.conf.cpu().numpy()
            if pr.boxes is not None:
                for b in pr.boxes:
                    person_boxes_raw.append(list(map(float, b.xyxy[0])))

            # Dibujar TODOS los esqueletos detectados
            for p_idx in range(len(kps_data)):
                draw_skeleton(annotated, kps_data[p_idx], conf_data[p_idx],
                              highlight_arms=True)

            # Clasificar persona más cercana al arma
            nearest = find_nearest_person(weapon_boxes[0][:4], person_boxes_raw)
            if nearest is not None and nearest < len(kps_data):
                angles     = get_arm_angles(kps_data[nearest], conf_data[nearest])
                pose_label = classify_pose(angles)
                draw_pose_label(annotated, kps_data[nearest], conf_data[nearest], pose_label)

        # --- Stage 3: filtro temporal ---
        pose_window.append(pose_label)
        aiming_in_w = sum(1 for p in pose_window if p == 'AIMING')
        if aiming_in_w >= AIMING_RUN_THR:
            threat_confirmed = True

    # --- HUD + overlay de amenaza ---
    draw_hud(annotated, frame_i, max_f, n_weapon_frames,
             pose_label, aiming_in_w, threat_confirmed, out_w)
    draw_threat_overlay(annotated, threat_confirmed, w_src, h_src)

    # Reescalar y escribir
    resized = cv2.resize(annotated, (out_w, out_h))
    writer.write(resized)

    if frame_i % 50 == 0:
        print(f'  frame {frame_i}/{max_f}  pose={pose_label}  threat={threat_confirmed}')

    frame_i += 1

cap.release()
writer.release()
os.remove(local_in)
print(f'OK Render completo — {frame_i} frames procesados')

Renderizando 225 frames (9.0s) -> 1280x960 @ 25fps
  frame 0/225  pose=NEUTRAL  threat=False
  frame 50/225  pose=NEUTRAL  threat=True
  frame 100/225  pose=AIMING  threat=True
  frame 150/225  pose=HOLDING  threat=True
  frame 200/225  pose=NEUTRAL  threat=True
OK Render completo — 225 frames procesados


In [16]:
# ============================================================
# RE-ENCODE CON FFMPEG (H.264) Y GUARDAR EN DRIVE
# mp4v de OpenCV no es compatible con todos los reproductores.
# ffmpeg convierte a H.264 estándar.
# ============================================================

final_out = f'{OUT_DIR}/demo_{CLIP_ID}.mp4'
cmd = (
    f'ffmpeg -y -i {local_out} '
    f'-vcodec libx264 -crf 20 -preset fast '
    f'-pix_fmt yuv420p '
    f'"{final_out}" '
    f'-loglevel error'
)
ret = os.system(cmd)
if ret == 0:
    os.remove(local_out)
    size_mb = Path(final_out).stat().st_size / 1024 / 1024
    print(f'OK Vídeo guardado en: {final_out}  ({size_mb:.1f} MB)')
else:
    print(f'ffmpeg falló (ret={ret}). El vídeo sin recodificar está en: {local_out}')
    print('Puedes descargarlo manualmente desde /content/')

OK Vídeo guardado en: /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/demo/demo_PAH1_C1_P2_V1_HB_3.mp4  (2.1 MB)
